<!-- nav -->
[← Implementing a module](02_Implementing_A_Module.ipynb) · [Home](../../README.md) · [Checking and saving →](04_Checking_And_Saving.ipynb)

# Organizing a design

An implementation's structure is fixed by what it holds: its instances and channels. People also organize a design in
ways that are not structure: the stages of a pipeline, its clock domains, the pages of a diagram. Those groupings
overlap, and changing one changes nothing about what runs.

`Structure` models both. A *container* (`Structure.Container`) holds what it contains through `Structure.Contains`.
Containment is *hard* (structure) or *soft* (organization), and the container decides which: a container that another
container *organizes* (`Structure.Organizes(hard, soft)`) is a *soft container*, and everything it contains, it
contains softly. Implementations and instances are containers.

In [1]:
from mbse.Schemas.Framework import Builders
from mbse.Architecture.Systems.Examples import Pipeline
from mbse.Architecture.Systems.Framework import Interactions, Structure

store = Builders.Store()
p = Pipeline.build(store)
I, S = p.schemas, p.structure
print(I.Implementation.refines.name, I.Instance.refines.name)

Structure.Container Structure.Container


## Hard containment is structure

An object has at most one hard container, and no container contains itself. An implementation holds its instances and
channels hard: `Interactions.Check` counts those relations as hard containment, beside `Structure.Contains`.

In [2]:
sensor = p.instance["sensor"]
print([Interactions.path(c) for c in Interactions.Check().hard_containers(sensor)])
print(Structure.hard_containers(sensor))  # Structure.Contains alone: none

['reference']
[]


## Soft containers group what their hard container holds

The example's implementation organizes two soft containers, `acquisition` and `processing`, which group its instances
and channels by pipeline stage. Each lies along the axis `dataflow` (`Structure.Orients(axis, soft_container)`): an
*axis* is a direction of organization, and soft containers along it are its steps.

In [3]:
for entry in p.impl.organizes:
    soft = entry.soft
    members = [Interactions.path(e.c2) for e in soft.contents]
    axis = [str(e.axis.name) for e in soft.axis]
    print(f"{soft.name}: along {axis}, holds {members}")

acquisition: along ['dataflow'], holds ['reference.sensor', 'reference.raw']
processing: along ['dataflow'], holds ['reference.filter', 'reference.logger', 'reference.filtered']


An object may lie in several soft containers, along different axes. Here a second axis, `clock`, splits the same
implementation into clock domains that cut across the stages: the filter is in the `processing` stage and in the
`fast` domain.

In [4]:
with store.transaction():
    clock = store.Structure.Axis(lambda a: a.name("clock").description("Clock domains"))
    for name, members in (("fast", [p.instance["filter"], p.channel["raw"]]),
                          ("slow", [p.instance["sensor"], p.instance["logger"], p.channel["filtered"]])):
        store.Structure.Container(lambda c, name=name, members=members: (
            c
            .name(name)
            .organizer(lambda a: a.insert(lambda e: e.hard(p.impl)))
            .axis(lambda a: a.insert(lambda e: e.axis(clock)))
            .contents(lambda a: [a.insert(lambda e, m=m: e.c2(m)) for m in members])
        ))

filt = p.instance["filter"]
print([str(e.c1.name) for e in filt.containers])
print(Interactions.Check()(p.pipeline))

['processing', 'fast']
[]


## What a soft container may hold

A soft container lies inside one hard container, and groups only what that hard container holds. Grouping something
from elsewhere is a problem `Check` reports, as is an axis on a container nothing organizes.

In [5]:
with store.transaction():
    elsewhere = store.Structure.Container(lambda c: c.name("bench"))
    store.Structure.Container(p.sections["acquisition"], lambda c: c.contents(lambda a: a.insert(lambda e: e.c2(elsewhere))))
    store.Structure.Container(lambda c: c.name("stray").axis(lambda a: a.insert(lambda e: e.axis(clock))))
for problem in Interactions.Check()(p.pipeline):
    print(problem)

Structure.Container 'acquisition' contains Structure.Container 'bench' softly, but its hard container, Interactions.Implementation 'reference', does not contain it
Structure.Container 'stray' lies along an axis, but no container organizes it: only a soft container lies along an axis


Next, [Checking and saving](04_Checking_And_Saving.ipynb) reads every kind of problem, and saves and loads a design.

<!-- nav -->
[← Implementing a module](02_Implementing_A_Module.ipynb) · [Home](../../README.md) · [Checking and saving →](04_Checking_And_Saving.ipynb)